# MIFA: reconciling provider nomenclature with MARIS

> A repeatable procedure for mapping data provider names (nuclides, species, units, etc.) to MARIS standard identifiers.

Every data provider uses its own naming conventions. Caesium-137 might appear as Cs-137, CS137, cs137, Cs 137, or even CS134/137 for a combined measurement. Species names can be abbreviated, misspelled, or written in different languages. Units are expressed in various notations. Before any data can be encoded into the MARIS NetCDF format, these nomenclatures must be reconciled with the standard MARIS identifiers.

Nomenclature reconciliation cannot be fully automated. It is an inherently human process that requires domain expertise. Someone must look at the borderline cases and decide the correct mapping. But automation can handle the bulk work, leaving only the ambiguous cases for expert judgement.

The MIFA pattern breaks this reconciliation into four concrete steps: Match, Inspect, Fix, Apply. Each step has a clear outcome and a well-defined handover to the next.

In [ ]:
from marisco.match import fuzzy_merge, fix_lut, Lut, lut_from, uniq_across_dfs
import pandas as pd

## Step 1: Match

Let the computer do brute-force fuzzy matching. It is good at it. You give it the provider values and the MARIS reference lookup table, and it finds the closest match for each entry along with a distance score.

The distance score tells you how far apart two strings are. A score of 0 means an exact match. The higher the score, the more the strings differ. This score is your signal for which cases need human attention.

In [ ]:
# A small example: provider nuclide values against a MARIS reference table
provider_lut = pd.DataFrame({'value': ['cs137', 'cs134_137_tot', 'cs144', 'k40', 'cs134']})
maris_ref = pd.DataFrame({
    'maris_id': [1, 2, 3, 33],
    'name': ['cs137', 'k40', 'sr90', 'cs134_137_tot'],
})

merged = fuzzy_merge(provider_lut, maris_ref, left_on='value', right_on='name')
merged

## Step 2: Inspect

Look at the non-exact matches, those with a score above 0. These are the cases where the fuzzy match may have gotten it wrong. Your job is to review each one and decide whether the automatically matched MARIS name is correct.

In this example, cs134_137_tot matched to k40 with a high score of 12, clearly wrong. And cs134 and cs144 have no match at all because they are not in the MARIS reference table.

In [ ]:
# Filter to see only non-exact matches
merged[merged['score'] > 0][['value', 'name', 'maris_id', 'score']]

There are two kinds of problems you will encounter.

**False matches** - the algorithm picked a MARIS name, but it is the wrong one. For example, cs134_137_tot was matched to k40 because of some accidental string similarity. These need an override telling the system the correct target.

**Missing matches** - the provider uses a value that does not exist in the MARIS reference at all. For example, cs134 and cs144 are not standard MARIS nuclide names. You need to research what they actually represent and map them to the closest MARIS equivalent. In this case, cs144 is likely a historical typo for cs137, while cs134 might need to be mapped to cs134_137_tot or kept as a separate entry depending on the data provider's methodology.

## Step 3: Fix

Apply expert overrides for every case the fuzzy match got wrong. You write a dictionary that maps each problematic provider value to the correct MARIS name. The fix_lut function applies these overrides and resets the score to 0 for the fixed entries, so you can verify that no issues remain.

In [ ]:
# Expert overrides: provider value -> correct MARIS name
overrides = {
    'cs134_137_tot': 'cs134_137_tot',  # correct match exists
    'cs144': 'cs137',                   # typo for cs137
    'cs134': 'cs134_137_tot',           # combined measurement
}

fixed = fix_lut(merged, overrides, maris_ref,
                left_on='value', right_on='name', id_col='maris_id')
fixed[fixed['score'] > 0]

If all entries now have a score of 0, the lookup table is complete and ready for use.

In [ ]:
assert len(fixed[fixed['score'] > 0]) == 0, "Unresolved matches remain"
print("All entries resolved.")

## Step 4: Apply

Wrap the resolved lookup table in a callable Lut object. This lets you pass it directly into a callback or use it to transform a DataFrame column.

In [ ]:
lut = Lut(fixed, key_col='value', val_col='maris_id')
print(lut('cs137'))       # 1
print(lut('cs134_137_tot'))  # 33

## Two common scenarios

The MIFA pattern works for two situations that handlers encounter regularly.

### Scenario A: Provider supplies an explicit nomenclature table

Some providers include a separate file that maps their codes to scientific names. For example a RUBIN_NAME.csv file that maps species codes to their full scientific names. In this case you load that file directly as your provider_lut and run the MIFA pipeline on it.

The fuzzy_merge function matches the provider's scientific names against the MARIS reference. You inspect the borderline scores, fix any mismatches with overrides, and create the Lut. The HELCOM handler uses this approach for species names.

### Scenario B: Provider does not supply a nomenclature

Many providers only share measurement data without a mapping table. In this case you derive the unique values directly from the data columns using lut_from, which collects all unique entries across every sample group (SEAWATER, BIOTA, SEDIMENT) into a single DataFrame.

Once you have this derived lookup table, the MIFA pipeline proceeds exactly as in Scenario A: match against the MARIS reference, inspect the borderline scores, fix with overrides, and create the Lut.

In [ ]:
# Scenario B in action: deriving unique values from provider data
provider_data = {
    'SEAWATER': pd.DataFrame({'NUCLIDE': ['cs137', 'cs134', 'cs137', 'k40']}),
    'BIOTA':    pd.DataFrame({'NUCLIDE': ['cs137', 'k40', 'sr90', 'cs134_137_tot']}),
}

provider_lut = lut_from(provider_data, 'NUCLIDE')
provider_lut

In [ ]:
# Complete MIFA pipeline on the derived lookup table
maris_ref = pd.DataFrame({
    'maris_id': [1, 2, 3, 33],
    'name': ['cs137', 'k40', 'sr90', 'cs134_137_tot'],
})

merged = fuzzy_merge(provider_lut, maris_ref, left_on='value', right_on='name')
overrides = {'cs134_137_tot': 'cs134_137_tot', 'cs134': 'cs134_137_tot'}
fixed = fix_lut(merged, overrides, maris_ref,
                left_on='value', right_on='name', id_col='maris_id')
lut = Lut(fixed, key_col='value', val_col='maris_id')

assert lut('cs137') == 1
assert lut('cs134_137_tot') == 33
print("All assertions pass.")

## Using MIFA in a handler

Within a handler notebook, the MIFA pattern typically appears as a self-contained pipeline section. The handler documents the decisions specific to that provider: the overrides dictionary, any data-quality issues discovered during inspection, and any feedback sent to the data provider. A handler does not need to re-explain the MIFA pattern itself. It can simply link to this guide and say "we follow the MIFA pattern."

The API reference for the individual functions (fuzzy_merge, fix_lut, Lut, lut_from) is in the match API module, which also documents the deprecated Remapper class kept for backward compatibility.